# RESULTS — mPFC Multi-session pipeline
## Améliorations vs version précédente
- **5 méthodes CD** : projection, cosine, euclidean, matrix_cosine, matrix_euclidean (+ LOO cross-validation pour les méthodes matricielles)
- **Threshold optimal** : `argmax(P(lick|stim_det) - P(lick|ITI_det))` — critère comportemental (vs Youden simple)
- **EngagedTrials cutoff** : session tronquée à la fin de la période engagée
- **Extension threshold** : couverture jusqu'au max réel du score (au-delà du 99.9ème percentile)
- **`detect_peaks_fn` robuste** : filtre `np.isfinite`, refractory en `ceil`
- **`p_lick_curve` & `burst_density`** : nouvelles métriques dans chaque résultat
- **Garde-fous plots** : plus de crash si `results` est vide

In [48]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.stats import wilcoxon
from pynwb import NWBHDF5IO
import ipywidgets as widgets
from IPython.display import display

# ── Paramètres globaux ──────────────────────────────────────────────────────
THIS_AREA     = 'mPFC'
Bin_size      = 0.01
AP_Bin_Res    = 1.0 / Bin_size
pre_bins      = 50
post_bins     = 100
amps_stim     = [1, 2, 3, 4]
tol_post      = 0.1   # Fenêtre post-stimulus élargie (mPFC : réponses plus lentes)
refractory_s  = 0.5   # Réfractaire augmenté (évite de fragmenter l'activité diffuse)
lick_window_s = 1.0
N_PCT         = 200
pct_axis      = np.linspace(1, 99.9, N_PCT)  # Hauts percentiles (mPFC très bruité)
time_axis     = (np.arange(pre_bins + post_bins + 1) - pre_bins) * Bin_size
colors_amp    = {1: 'gold', 2: 'orange', 3: 'tomato', 4: 'red'}

# ── Paramètres Coding Direction ─────────────────────────────────────────────
CD_USE_AMP        = 4
CD_METHOD         = 'matrix_cosine'  # 'projection' | 'cosine' | 'euclidean' | 'matrix_cosine' | 'matrix_euclidean'
CD_WINDOW         = (55, 80)      # Fenêtre CD vectorielle (100–300 ms post-stim, adapté mPFC)
CD_WINDOWS_2      = [(4, 10), (12, 30)]  # Fenêtres matricielles
CD_PRE_BASELINE   = 10
CD_MATRIX_STEP_BINS = 1
CD_MATRIX_ZSCORE  = True

CI95 = lambda arr, axis=0: 1.96 * np.std(arr, axis=axis, ddof=1) / np.sqrt(arr.shape[axis])
print('Imports OK')

Imports OK


In [49]:
import glob, os

VALID_DIR     = 'data/valid/WR+'
SESSION_PATHS = sorted(glob.glob(os.path.join(VALID_DIR, '*.nwb')))

print(f'{len(SESSION_PATHS)} session(s) trouvée(s) dans {VALID_DIR}/ :')
for p in SESSION_PATHS:
    print(f'  {p}')

9 session(s) trouvée(s) dans data/valid/WR+/ :
  data/valid/WR+/AO026_20181122_180943.nwb
  data/valid/WR+/AO027_20181101_134512.nwb
  data/valid/WR+/AO028_20181102_110339.nwb
  data/valid/WR+/AO036_20190507_183647.nwb
  data/valid/WR+/AO039_20190626_160524.nwb
  data/valid/WR+/AO040_20190708_124008.nwb
  data/valid/WR+/AO049_20191009_160435.nwb
  data/valid/WR+/AO050_20191007_195438.nwb
  data/valid/WR+/AO051_20191004_182709.nwb


In [50]:
# ── Helpers ────────────────────────────────────────────────────────────────

def event_triggered_mean(M, event_times, res, pre=50, post=100):
    W   = pre + post + 1
    acc = np.zeros((M.shape[0], W), dtype=np.float32)
    n   = 0
    for ev in event_times:
        pt = int(np.round(ev * res))
        if pt - pre < 0 or pt + post >= M.shape[1]:
            continue
        acc += M[:, pt - pre: pt + post + 1]
        n   += 1
    return acc / max(n, 1), n

def detect_peaks_fn(score, t, thr, refrac, bin_s, t0, t1):
    """Détection de pics robuste : filtre NaN/Inf, refractory en ceil."""
    score = np.asarray(score)
    t     = np.asarray(t)
    mask  = np.isfinite(score) & (t >= t0) & (t <= t1) & (score > thr)
    idx   = np.where(mask)[0]
    if idx.size == 0:
        return np.array([], dtype=float)
    cuts   = np.where(np.diff(idx) > 1)[0] + 1
    groups = np.split(idx, cuts)
    peaks  = np.array([g[np.argmax(score[g])] for g in groups], dtype=int)
    min_b  = max(1, int(np.ceil(refrac / bin_s)))
    keep   = [peaks[0]]
    for p in peaks[1:]:
        if p - keep[-1] >= min_b:
            keep.append(p)
        elif score[p] > score[keep[-1]]:
            keep[-1] = p
    return t[np.array(keep, dtype=int)]

def _normalize_vec(v):
    return v / (np.linalg.norm(v) + 1e-12)

def zscore_vector(v, eps=1e-10):
    v  = np.asarray(v, dtype=np.float32)
    mu = np.mean(v)
    sd = np.std(v)
    return (v - mu) / (sd + eps)

def flatten(mat):
    return mat.reshape(-1).astype(np.float32)

# ── CD vectorielle simple ───────────────────────────────────────────────────
def build_cd_vector(STM_by_amp, CATCH_M, amp=4, win=(55, 80)):
    i0, i1 = win
    cd = STM_by_amp[amp][:, i0:i1].mean(axis=1) - CATCH_M[:, i0:i1].mean(axis=1)
    return _normalize_vec(cd)

# ── CD matrix / template ───────────────────────────────────────────────────
def extract_two_window_matrix(AP, event_time, resolution, win1, win2, pre_baseline):
    pt = int(np.round(event_time * resolution))
    if pt - pre_baseline < 0 or pt + max(win1[1], win2[1]) >= AP.shape[1]:
        return None
    seg1 = AP[:, pt + win1[0]: pt + win1[1]]
    seg2 = AP[:, pt + win2[0]: pt + win2[1]]
    if seg1.shape[1] == 0 or seg2.shape[1] == 0:
        return None
    mat      = np.concatenate([seg1, seg2], axis=1)
    baseline = AP[:, pt - pre_baseline: pt].mean(axis=1, keepdims=True)
    return (mat - baseline).astype(np.float32)

def build_stack(AP, times, resolution, win1, win2, pre_baseline):
    mats, kept = [], []
    for t in times:
        m = extract_two_window_matrix(AP, t, resolution, win1, win2, pre_baseline)
        if m is not None:
            mats.append(m); kept.append(t)
    if not mats:
        return None, np.array([])
    return np.stack(mats, axis=0), np.array(kept)

def _build_loo_ref(stim_stack, catch_stack, exclude_i, zscore_flag):
    n = stim_stack.shape[0]
    if exclude_i >= 0 and n > 1:
        keep      = [j for j in range(n) if j != exclude_i]
        mean_stim = stim_stack[keep].mean(axis=0)
    else:
        mean_stim = stim_stack.mean(axis=0)
    if catch_stack is not None and len(catch_stack) > 0:
        cd = mean_stim - catch_stack.mean(axis=0)
    else:
        cd = mean_stim.copy()
    ref = flatten(cd)
    if zscore_flag:
        ref = zscore_vector(ref)
    return ref

def _score_vec(vec, ref_vector, metric, zscore_flag):
    if metric in ('euclidean', 'euclidian'):
        ref_n = ref_vector / (np.linalg.norm(ref_vector) + 1e-10)
        return float(-np.linalg.norm(vec - ref_n))
    elif metric == 'cosine':
        if zscore_flag:
            vec = zscore_vector(vec)
        denom = (np.linalg.norm(vec) * np.linalg.norm(ref_vector)) + 1e-10
        return float(np.dot(vec, ref_vector) / denom)
    else:
        raise ValueError(f'Unknown matrix metric: {metric}')

def compute_matrix_cd_score(AP_Bin, Stim_by_amp, Catch_times, res,
                             amp=4, wins=((0, 4), (5, 19)),
                             pre_baseline=10, metric='cosine',
                             zscore=True, step_bins=1):
    """Score CD matriciel avec leave-one-out cross-validation."""
    win1, win2 = tuple(wins[0]), tuple(wins[1])
    step_bins  = max(1, int(step_bins))
    max_win    = max(win1[1], win2[1])

    stim_stack,  stim_kept  = build_stack(AP_Bin, Stim_by_amp[amp], res, win1, win2, pre_baseline)
    catch_stack, catch_kept = build_stack(AP_Bin, Catch_times,       res, win1, win2, pre_baseline)

    if stim_stack is None or len(stim_kept) == 0:
        raise ValueError(f'No valid stim windows for matrix CD (amp={amp})')

    stim_pts = np.array([int(np.round(t * res)) for t in stim_kept])
    full_ref  = _build_loo_ref(stim_stack, catch_stack, -1, zscore)
    loo_refs  = [_build_loo_ref(stim_stack, catch_stack, i, zscore)
                 for i in range(len(stim_kept))]

    scores, valid_pts = [], []
    T = AP_Bin.shape[1]

    for pt in range(pre_baseline, T - max_win, step_bins):
        mat = extract_two_window_matrix(AP_Bin, pt / res, res, win1, win2, pre_baseline)
        if mat is None:
            continue
        vec = flatten(mat)
        trial_idx = -1
        for i, spt in enumerate(stim_pts):
            if spt <= pt < spt + max_win:
                trial_idx = i
                break
        ref = loo_refs[trial_idx] if trial_idx >= 0 else full_ref
        scores.append(_score_vec(vec, ref, metric, zscore))
        valid_pts.append(pt)

    score = np.asarray(scores, dtype=np.float32)
    t_sig = np.asarray(valid_pts, dtype=np.float32) * Bin_size

    if catch_stack is not None and len(catch_stack) > 0:
        cd_matrix  = stim_stack.mean(axis=0) - catch_stack.mean(axis=0)
        catch_mode = 'stim_minus_catch'
    else:
        cd_matrix  = stim_stack.mean(axis=0)
        catch_mode = 'none'

    meta = {
        'kind': 'matrix_template', 'cd': cd_matrix, 'ref_vector': full_ref,
        'method': f'matrix_{metric}', 'wins': (win1, win2),
        'pre_baseline': pre_baseline, 'zscore': zscore,
        'n_stim_windows': len(stim_kept),
        'n_catch_windows': len(catch_kept) if catch_stack is not None else 0,
        'catch_mode': catch_mode, 'step_bins': step_bins, 'cross_validated': True,
    }
    return score, t_sig, meta

# ── Interface commune ───────────────────────────────────────────────────────
def compute_cd_score(AP_Bin, STM_by_amp, CATCH_M, Stim_by_amp, Catch_times, res,
                     method='projection', amp=4, win=(55, 80),
                     wins=((0, 4), (5, 19)), pre_baseline=10,
                     matrix_step_bins=1, matrix_zscore=True):
    eps = 1e-12
    if method == 'projection':
        cd    = build_cd_vector(STM_by_amp, CATCH_M, amp=amp, win=win)
        score = cd @ AP_Bin
        t_sig = np.arange(AP_Bin.shape[1], dtype=np.float32) * Bin_size
        return score, t_sig, {'kind': 'vector', 'cd': cd, 'method': method, 'win': win}
    if method == 'cosine':
        cd    = build_cd_vector(STM_by_amp, CATCH_M, amp=amp, win=win)
        x_norm = np.linalg.norm(AP_Bin, axis=0) + eps
        score  = (cd @ AP_Bin) / x_norm
        t_sig  = np.arange(AP_Bin.shape[1], dtype=np.float32) * Bin_size
        return score, t_sig, {'kind': 'vector', 'cd': cd, 'method': method, 'win': win}
    if method in ('euclidean', 'euclidian'):
        cd    = build_cd_vector(STM_by_amp, CATCH_M, amp=amp, win=win)
        diff  = AP_Bin - cd[:, None]
        score = -np.linalg.norm(diff, axis=0)
        t_sig = np.arange(AP_Bin.shape[1], dtype=np.float32) * Bin_size
        return score, t_sig, {'kind': 'vector', 'cd': cd, 'method': method, 'win': win}
    if method == 'matrix_cosine':
        return compute_matrix_cd_score(AP_Bin, Stim_by_amp, Catch_times, res,
            amp=amp, wins=wins, pre_baseline=pre_baseline,
            metric='cosine', zscore=matrix_zscore, step_bins=matrix_step_bins)
    if method in ('matrix_euclidean', 'matrix_euclidian'):
        return compute_matrix_cd_score(AP_Bin, Stim_by_amp, Catch_times, res,
            amp=amp, wins=wins, pre_baseline=pre_baseline,
            metric='euclidean', zscore=True, step_bins=matrix_step_bins)
    raise ValueError(f'Unknown CD method: {method}')

print('Helpers OK')

Helpers OK


In [51]:
# ── Pipeline par session ────────────────────────────────────────────────────

def run_session(path, cd_method=None):
    if cd_method is None:
        cd_method = CD_METHOD
    print(f"\n{'='*60}\nSESSION : {path}")

    # 1. Chargement NWB ────────────────────────────────────────────────────
    with NWBHDF5IO(path, 'r') as io:
        nwb  = io.read()
        tr   = nwb.trials.to_dataframe().copy()
        areas = nwb.units['Target_area'].data[:]
        areas = np.array([a.decode() if isinstance(a, (bytes, bytearray)) else a for a in areas])
        mask  = (areas == THIS_AREA)
        spike_times = [np.asarray(nwb.units['spike_times'][i]) for i in np.where(mask)[0]]

        try:
            beh        = nwb.processing['behavior']
            piezo_obj  = beh['BehavioralTimeSeries']['PiezoLickSignal']
            piezo_data = np.array(piezo_obj.data[:], dtype=np.float32)
            piezo_ts   = np.array(piezo_obj.timestamps[:])
            has_piezo  = True
        except Exception:
            has_piezo = False
            print('  [!] Pas de piezo')

        # EngagedTrials cutoff ─────────────────────────────────────────────
        try:
            et_obj  = nwb.processing['behavior']['BehavioralEvents']['EngagedTrials']
            et_ts   = np.array(et_obj.timestamps[:])
            et_data = np.array(et_obj.data[:])
            t_max   = float(et_ts[et_data == 1].max()) if (et_data == 1).any() else np.inf
        except Exception:
            t_max = np.inf

    if t_max < np.inf:
        print(f'  [EngagedTrials] trimming à t={t_max:.1f}s  (session complète: {float(tr["stop_time"].max()):.1f}s)')
        tr          = tr[tr['start_time'] < t_max].copy()
        spike_times = [st[st < t_max] for st in spike_times]
        if has_piezo:
            pm         = piezo_ts < t_max
            piezo_data = piezo_data[pm]
            piezo_ts   = piezo_ts[pm]

    print(f'  {THIS_AREA} units : {len(spike_times)}')
    if len(spike_times) == 0:
        return None

    # 2. AP_Bin ────────────────────────────────────────────────────────────
    t0_s   = float(tr['start_time'].min())
    t1_s   = float(tr['stop_time'].max())
    n_bins = int(np.round(t1_s * AP_Bin_Res))
    edges  = np.linspace(0.0, n_bins * Bin_size, n_bins + 1)
    AP_Bin = np.zeros((len(spike_times), n_bins), dtype=np.float32)
    for k, st in enumerate(spike_times):
        counts, _ = np.histogram(st, bins=edges)
        AP_Bin[k] = counts

    # 3. PSTH stim par amp + catch ─────────────────────────────────────────
    stim_sel    = tr['whisker_stim_time'].notna() & tr['whisker_stim_amplitude'].isin(amps_stim)
    Stim_by_amp = {
        a: np.sort(tr.loc[stim_sel & (tr['whisker_stim_amplitude'] == a), 'whisker_stim_time'].to_numpy())
        for a in amps_stim
    }
    if 'no_stim' in tr.columns and 'no_stim_time' in tr.columns:
        catch_sel   = tr['no_stim'].astype(bool) & tr['no_stim_time'].notna()
        Catch_times = np.sort(tr.loc[catch_sel, 'no_stim_time'].to_numpy())
    else:
        Catch_times = np.array([])

    CATCH_M, _ = event_triggered_mean(AP_Bin, Catch_times, AP_Bin_Res, pre_bins, post_bins)
    STM_by_amp = {}
    for a in amps_stim:
        STM_by_amp[a], _ = event_triggered_mean(AP_Bin, Stim_by_amp[a], AP_Bin_Res, pre_bins, post_bins)

    psth_stim  = {a: STM_by_amp[a].mean(axis=0) / Bin_size for a in amps_stim}
    psth_catch = CATCH_M.mean(axis=0) / Bin_size

    # 4. Coding Direction ──────────────────────────────────────────────────
    score, t_sig, cd_meta = compute_cd_score(
        AP_Bin, STM_by_amp, CATCH_M, Stim_by_amp, Catch_times, AP_Bin_Res,
        method=cd_method, amp=CD_USE_AMP,
        win=CD_WINDOW, wins=CD_WINDOWS_2,
        pre_baseline=CD_PRE_BASELINE,
        matrix_step_bins=CD_MATRIX_STEP_BINS,
        matrix_zscore=CD_MATRIX_ZSCORE,
    )

    # 5. Lick detection ────────────────────────────────────────────────────
    if has_piezo:
        piezo_rate = 1.0 / np.median(np.diff(piezo_ts))
        piezo_env  = gaussian_filter1d(np.abs(piezo_data), sigma=int(0.01 * piezo_rate))
        median_env = np.median(piezo_env)
        mad_env    = np.median(np.abs(piezo_env - median_env))
        thr_piezo  = median_env + 15.0 * mad_env
        min_dist   = int(0.15 * piezo_rate)
        peak_idx, _ = find_peaks(piezo_env, height=thr_piezo, distance=min_dist)
        all_lick_times = piezo_ts[peak_idx]
        print(f'  Licks : {len(all_lick_times)}')
    else:
        all_lick_times = np.array([])

    # 6. TP / FP curves + critère comportemental ───────────────────────────
    in_range = (t_sig >= t0_s) & (t_sig <= t1_s)
    score_in = score[in_range]
    ths_pct  = np.percentile(score_in, pct_axis)
    # Extension jusqu'au vrai max du score (au-delà du 99.9ème percentile)
    if score_in.max() > ths_pct[-1]:
        ths_ext = np.linspace(ths_pct[-1], score_in.max(), 20)[1:]
        ths = np.concatenate([ths_pct, ths_ext])
    else:
        ths = ths_pct

    all_stim_times = np.sort(np.concatenate([Stim_by_amp[a] for a in amps_stim]))
    all_events     = np.sort(np.concatenate([all_stim_times, Catch_times]))
    iti_dur        = max((t1_s - t0_s) - len(all_events) * 1.0, 1.0)

    tp_c, fp_c, fp_catch_c = {a: [] for a in amps_stim}, [], []
    p_lick_curve_list, burst_density_list = [], []

    for thr in ths:
        det   = detect_peaks_fn(score, t_sig, thr, refractory_s, Bin_size, t0_s, t1_s)
        n_det = max(len(det), 1)
        in_win = np.zeros(len(det), dtype=bool)
        for st in all_stim_times:
            in_win |= (det >= st) & (det <= st + tol_post)
        fp_c.append(np.sum(~in_win) / n_det)
        fp_catch_c.append(
            sum(np.any((det >= st) & (det <= st + tol_post)) for st in Catch_times)
            / max(len(Catch_times), 1)
        )
        for a in amps_stim:
            tp_c[a].append(
                sum(np.any((det >= st) & (det <= st + tol_post)) for st in Stim_by_amp[a])
                / max(len(Stim_by_amp[a]), 1)
            )
        # Critère comportemental : P(lick|stim_det) - P(lick|ITI_det)
        if len(det) > 0 and len(all_lick_times) > 0:
            stim_det = np.array([dt for dt in det
                                 if np.any((dt >= all_stim_times) & (dt <= all_stim_times + tol_post))])
            iti_det  = np.array([dt for dt in det
                                 if not np.any((dt >= all_events) & (dt <= all_events + 1.0))])
            p_ls = float(np.mean([np.any((all_lick_times >= dt) & (all_lick_times <= dt + lick_window_s))
                                  for dt in stim_det])) if len(stim_det) >= 2 else np.nan
            p_li = float(np.mean([np.any((all_lick_times >= dt) & (all_lick_times <= dt + lick_window_s))
                                  for dt in iti_det]))  if len(iti_det)  >= 2 else np.nan
            p_lick_curve_list.append(p_ls - p_li if not (np.isnan(p_ls) or np.isnan(p_li)) else np.nan)
        else:
            p_lick_curve_list.append(np.nan)
        # Burst density (ITI dets / minute)
        if len(det) > 0:
            burst_mask = np.array([not np.any((dt >= all_events) & (dt <= all_events + 1.0)) for dt in det])
            burst_density_list.append(float(burst_mask.sum() / (iti_dur / 60.0)))
        else:
            burst_density_list.append(0.0)

    tp_c = {a: np.array(tp_c[a]) for a in amps_stim}
    fp_c, fp_catch_c = np.array(fp_c), np.array(fp_catch_c)
    p_lick_curve  = np.array(p_lick_curve_list)
    burst_density = np.array(burst_density_list)

    # 7. Meilleur threshold ────────────────────────────────────────────────
    # Critère comportemental en priorité ; fallback sur Youden si insuffisant
    valid_lick = ~np.isnan(p_lick_curve)
    if valid_lick.sum() >= 3:
        lick_best_i = int(np.nanargmax(p_lick_curve))
    else:
        lick_best_i = None

    youden   = tp_c[4] - fp_c
    mask_50  = tp_c[4] >= 0.5
    mask_any = tp_c[4] > 0
    if lick_best_i is not None:
        best_i = lick_best_i
        print(f'  Threshold sélectionné par critère comportemental (P_lick_diff)')
    elif mask_50.any():
        best_i = int(np.argmax(np.where(mask_50,  youden, -np.inf)))
    elif mask_any.any():
        best_i = int(np.argmax(np.where(mask_any, youden, -np.inf)))
    else:
        best_i = 0

    best_thr = ths[best_i]
    det_best = detect_peaks_fn(score, t_sig, best_thr, refractory_s, Bin_size, t0_s, t1_s)
    print(f'  Best thr={best_thr:.4f}  TP4={tp_c[4][best_i]:.2f}  FP={fp_c[best_i]:.2f}')

    # 8. Matrice stim/burst × lick/no lick ────────────────────────────────
    stim_results = {}
    for a in amps_stim:
        tl, tnl, ml, mnl = 0, 0, 0, 0
        for st in Stim_by_amp[a]:
            det  = np.any((det_best >= st) & (det_best <= st + tol_post))
            lick = np.any((all_lick_times >= st) & (all_lick_times <= st + lick_window_s))
            if   det  and lick:      tl  += 1
            elif det  and not lick:  tnl += 1
            elif not det and lick:   ml  += 1
            else:                    mnl += 1
        stim_results[a] = (tl, tnl, ml, mnl)

    # Bursts spontanés (ITI strict, exclusion 1.5 s pour mPFC)
    burst_times = np.array([dt for dt in det_best
                             if not np.any((dt >= all_events) & (dt <= all_events + 1.5))])
    burst_lick   = sum(np.any((all_lick_times >= dt) & (all_lick_times <= dt + lick_window_s))
                       for dt in burst_times)
    burst_nolick = len(burst_times) - burst_lick

    # 9. Chance level ──────────────────────────────────────────────────────
    n_ctrl = max(len(burst_times), 1)
    rng    = np.random.default_rng(0)
    ctrl, n_tries = [], 0
    while len(ctrl) < n_ctrl and n_tries < 500000:
        t = rng.uniform(t0_s + 1.0, t1_s - lick_window_s)
        near_trial  = np.any((t >= all_events) & (t <= all_events + 1.0))
        near_detect = np.any((det_best >= t - 1.0) & (det_best <= t + lick_window_s))
        if not near_trial and not near_detect:
            ctrl.append(t)
        n_tries += 1
    ctrl = np.array(ctrl)
    chance_lick = np.mean([np.any((all_lick_times >= t) & (all_lick_times <= t + lick_window_s))
                           for t in ctrl]) if len(ctrl) > 0 else np.nan
    print(f'  Bursts (ITI)={len(burst_times)}  Ctrl (same N, ITI)={len(ctrl)}')

    # 10. Bouts de lick par catégorie ──────────────────────────────────────
    bout_durs = {c: [] for c in ['stim4','stim3','stim2','stim1','burst','none']}
    if len(all_lick_times) > 1:
        lts   = np.sort(all_lick_times)
        cuts  = np.where(np.diff(lts) > 1.0)[0] + 1
        bouts = np.split(lts, cuts)
        b_s   = np.array([g[0]  for g in bouts])
        b_e   = np.array([g[-1] for g in bouts])
        for i, tb in enumerate(b_s):
            dur   = b_e[i] - tb
            cands = det_best[(det_best >= tb - lick_window_s) & (det_best < tb)]
            if len(cands) == 0:
                bout_durs['none'].append(dur); continue
            best_amp, has_burst = None, False
            for pk in cands:
                for a in [4, 3, 2, 1]:
                    if np.any((pk >= Stim_by_amp[a]) & (pk <= Stim_by_amp[a] + tol_post)):
                        if best_amp is None or a > best_amp: best_amp = a
                        break
                else:
                    has_burst = True
            if best_amp is not None: bout_durs[f'stim{best_amp}'].append(dur)
            else:                    bout_durs['burst'].append(dur)

    tp4_l, tp4_nl, _, _ = stim_results[4]
    p_stim4 = tp4_l / max(tp4_l + tp4_nl, 1)
    p_burst = burst_lick / max(len(burst_times), 1)
    print(f'  P(lick|stim4 det)={p_stim4:.3f}  P(lick|burst)={p_burst:.3f}  Chance={chance_lick:.3f}')

    return {
        'path'          : path,
        'n_units'       : len(spike_times),
        'cd_method'     : cd_method,
        'cd_meta'       : cd_meta,
        'psth_stim'     : psth_stim,
        'psth_catch'    : psth_catch,
        'thresholds'    : ths,
        'best_i'        : best_i,
        'tp_curves'     : tp_c,
        'fp_cont'       : fp_c,
        'fp_catch'      : fp_catch_c,
        'p_lick_curve'  : p_lick_curve,
        'burst_density' : burst_density,
        'stim_results'  : stim_results,
        'burst_lick'    : burst_lick,
        'burst_nolick'  : burst_nolick,
        'n_burst'       : len(burst_times),
        'chance_lick'   : chance_lick,
        'p_stim4'       : p_stim4,
        'p_burst'       : p_burst,
        'bout_durs'     : bout_durs,
    }

print('Pipeline OK')

Pipeline OK


In [52]:
# ── Lancement sur toutes les sessions ──────────────────────────────────────
results = []
for path in SESSION_PATHS:
    r = run_session(path)
    if r is not None:
        results.append(r)

N = len(results)
print(f'\n{N} session(s) traitée(s) avec succès')
for r in results:
    print(f"  {r['path'].split('/')[-1]}  —  {r['n_units']} units  —  méthode: {r['cd_method']}")

if N == 0:
    print('\n⚠️  Aucune session chargée.')
    print(f'   Vérifier que des fichiers .nwb existent dans : data/valid/')
    import os; print(f'   Répertoire courant : {os.getcwd()}')


SESSION : data/valid/WR+/AO026_20181122_180943.nwb
  [EngagedTrials] trimming à t=3880.9s  (session complète: 3882.9s)
  mPFC units : 51
  Licks : 2554
  Threshold sélectionné par critère comportemental (P_lick_diff)
  Best thr=-0.1891  TP4=0.18  FP=0.99
  Bursts (ITI)=1352  Ctrl (same N, ITI)=1352
  P(lick|stim4 det)=1.000  P(lick|burst)=0.229  Chance=0.215

SESSION : data/valid/WR+/AO027_20181101_134512.nwb
  [EngagedTrials] trimming à t=4628.9s  (session complète: 5383.6s)
  mPFC units : 52
  Licks : 3147
  Threshold sélectionné par critère comportemental (P_lick_diff)
  Best thr=0.3076  TP4=0.03  FP=0.73
  Bursts (ITI)=11  Ctrl (same N, ITI)=11
  P(lick|stim4 det)=0.000  P(lick|burst)=0.000  Chance=0.273

SESSION : data/valid/WR+/AO028_20181102_110339.nwb
  [EngagedTrials] trimming à t=6502.8s  (session complète: 6504.8s)
  mPFC units : 46
  Licks : 2064
  Threshold sélectionné par critère comportemental (P_lick_diff)
  Best thr=0.3819  TP4=0.02  FP=0.60
  Bursts (ITI)=3  Ctrl (sa

---
## Plots — Mean ± 95% CI across all sessions

In [53]:
# ── Plot 1 : Mean PSTH by amplitude + catch ───────────────────────────────
if not results:
    print('⚠️  Aucune session chargée — plots non disponibles.')
else:
    session_names_p1 = ['Mean'] + [r['path'].split('/')[-1] for r in results]

    def plot_psth(session='Mean'):
        fig, ax = plt.subplots(figsize=(10, 4))
        if session == 'Mean':
            catch_arr = np.stack([r['psth_catch'] for r in results], axis=0)
            m, ci = catch_arr.mean(axis=0), CI95(catch_arr)
            ax.plot(time_axis, m, color='steelblue', lw=2, label='Catch')
            ax.fill_between(time_axis, m - ci, m + ci, alpha=0.2, color='steelblue')
            for a in amps_stim:
                arr = np.stack([r['psth_stim'][a] for r in results], axis=0)
                m, ci = arr.mean(axis=0), CI95(arr)
                ax.plot(time_axis, m, color=colors_amp[a], lw=2, label=f'Amp {a}')
                ax.fill_between(time_axis, m - ci, m + ci, alpha=0.2, color=colors_amp[a])
            ax.set_title(f'Mean PSTH {THIS_AREA} — {N} sessions (mean ± 95% CI)')
        else:
            r = results[session_names_p1.index(session) - 1]
            ax.plot(time_axis, r['psth_catch'], color='steelblue', lw=2, label='Catch')
            for a in amps_stim:
                ax.plot(time_axis, r['psth_stim'][a], color=colors_amp[a], lw=2, label=f'Amp {a}')
            ax.set_title(f'PSTH — {session}')
        ax.axvline(0, color='k', linestyle='--', lw=1)
        ax.set_xlabel('Time relative to stim (s)')
        ax.set_ylabel('Firing rate (Hz)')
        ax.legend()
        plt.tight_layout()
        plt.show()

    widgets.interact(plot_psth, session=session_names_p1)

interactive(children=(Dropdown(description='session', options=('Mean', 'AO026_20181122_180943.nwb', 'AO027_201…

In [54]:
# ── Plot 2 : TP / FP vs Threshold ─────────────────────────────────────────
# Individual session → x-axis = absolute threshold value
# Mean              → per-session [0,1] normalisation + interpolation to common grid
if not results:
    print('⚠️  Aucune session chargée.')
else:
    session_names = ['Mean'] + [r['path'].split('/')[-1] for r in results]
    X_COMMON = np.linspace(0, 1, 300)

    def _interp_norm(r, key, amp=None):
        thr   = r['thresholds']
        thr_n = (thr - thr.min()) / (thr.max() - thr.min() + 1e-12)
        y     = r['tp_curves'][amp] if amp is not None else r[key]
        return np.interp(X_COMMON, thr_n, y)

    def plot_tpfp(session='Mean'):
        fig, ax = plt.subplots(figsize=(9, 5))
        if session == 'Mean':
            for r in results:
                thr   = r['thresholds']
                thr_n = (thr - thr.min()) / (thr.max() - thr.min() + 1e-12)
                for a in amps_stim:
                    ax.plot(thr_n, r['tp_curves'][a], color=colors_amp[a], lw=0.6, alpha=0.25)
                ax.plot(thr_n, r['fp_cont'],  color='black', lw=0.6, alpha=0.2, linestyle='--')
                ax.plot(thr_n, r['fp_catch'], color='gray',  lw=0.6, alpha=0.2, linestyle=':')
            for a in amps_stim:
                arr = np.stack([_interp_norm(r, None, amp=a) for r in results])
                m, ci = arr.mean(0), CI95(arr)
                ax.plot(X_COMMON, m, color=colors_amp[a], lw=2.5, label=f'TP amp {a}')
                ax.fill_between(X_COMMON, m - ci, m + ci, alpha=0.2, color=colors_amp[a])
            for label, key, color, ls in [('FP continuous', 'fp_cont', 'black', '--'),
                                           ('FP catch',      'fp_catch', 'gray',  ':')]:
                arr = np.stack([_interp_norm(r, key) for r in results])
                m, ci = arr.mean(0), CI95(arr)
                ax.plot(X_COMMON, m, color=color, lw=2.5, linestyle=ls, label=label)
                ax.fill_between(X_COMMON, m - ci, m + ci, alpha=0.15, color=color)
            best_norms = [(r['thresholds'][r['best_i']] - r['thresholds'].min()) /
                          (r['thresholds'].max() - r['thresholds'].min() + 1e-12) for r in results]
            mean_tp = np.mean([r['tp_curves'][4][r['best_i']] for r in results])
            mean_fp = np.mean([r['fp_cont'][r['best_i']] for r in results])
            ax.axvline(np.mean(best_norms), color='purple', lw=1.5, linestyle='--',
                       label=f'Best thr (mean)  TP4={mean_tp:.2f}  FP={mean_fp:.2f}')
            ax.set_xlabel('Threshold (normalised [0–1] per session)')
            ax.set_title(f'TP / FP vs Threshold — {N} sessions (mean ± 95% CI)')
        else:
            r   = results[session_names.index(session) - 1]
            thr = r['thresholds']
            for a in amps_stim:
                ax.plot(thr, r['tp_curves'][a], color=colors_amp[a], lw=2, label=f'TP amp {a}')
            ax.plot(thr, r['fp_cont'],  color='black', lw=2, linestyle='--', label='FP continuous')
            ax.plot(thr, r['fp_catch'], color='gray',  lw=2, linestyle=':', label='FP catch')
            if 'p_lick_curve' in r:
                ax2 = ax.twinx()
                valid = ~np.isnan(r['p_lick_curve'])
                ax2.plot(thr[valid], r['p_lick_curve'][valid], color='purple',
                         lw=1.5, linestyle=':', alpha=0.8, label='P_lick_diff')
                ax2.set_ylabel('P(lick|stim_det) − P(lick|ITI_det)', color='purple', fontsize=8)
                ax2.tick_params(axis='y', labelcolor='purple')
            best_val = thr[r['best_i']]
            tp_b     = r['tp_curves'][4][r['best_i']]
            fp_b     = r['fp_cont'][r['best_i']]
            ax.axvline(best_val, color='purple', lw=1.5, linestyle='--',
                       label=f'Best thr={best_val:.4f}  TP4={tp_b:.2f}  FP={fp_b:.2f}')
            ax.set_xlabel('Threshold (absolute value)')
            ax.set_title(f'TP / FP vs Threshold — {session}')
        ax.set_ylabel('Rate')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

    widgets.interact(plot_tpfp, session=session_names)

interactive(children=(Dropdown(description='session', options=('Mean', 'AO026_20181122_180943.nwb', 'AO027_201…

In [55]:
# ── Interactive visualisation : CD Score / Firing rate / Piezo ────────────
# Computation (AP_Bin, CD, score) is cached per session
from matplotlib.lines import Line2D

if not results:
    print('⚠️  Aucune session chargée.')
else:
    session_names_vis = [r['path'].split('/')[-1] for r in results]
    _vis_cache = {}

    def _compute_vis(session):
        path = next(r['path'] for r in results if r['path'].split('/')[-1] == session)
        print(f'  Computing {session}…')
        with NWBHDF5IO(path, 'r') as io:
            nwb   = io.read()
            tr    = nwb.trials.to_dataframe().copy()
            areas = nwb.units['Target_area'].data[:]
            areas = np.array([a.decode() if isinstance(a, (bytes, bytearray)) else a for a in areas])
            sp    = [np.asarray(nwb.units['spike_times'][i])
                     for i in np.where(areas == THIS_AREA)[0]]
            has_piezo, piezo_data, piezo_ts = False, None, None
            try:
                beh       = nwb.processing['behavior']
                pobj      = beh['BehavioralTimeSeries']['PiezoLickSignal']
                piezo_data = np.array(pobj.data[:], dtype=np.float32)
                piezo_ts   = np.array(pobj.timestamps[:])
                has_piezo  = True
            except Exception:
                pass
            try:
                et_obj  = nwb.processing['behavior']['BehavioralEvents']['EngagedTrials']
                et_ts   = np.array(et_obj.timestamps[:])
                et_data = np.array(et_obj.data[:])
                t_max   = float(et_ts[et_data == 1].max()) if (et_data == 1).any() else np.inf
            except Exception:
                t_max = np.inf
        if t_max < np.inf:
            tr = tr[tr['start_time'] < t_max].copy()
            sp = [s[s < t_max] for s in sp]
            if has_piezo:
                pm = piezo_ts < t_max
                piezo_data, piezo_ts = piezo_data[pm], piezo_ts[pm]
        t0_s   = float(tr['start_time'].min())
        t1_s   = float(tr['stop_time'].max())
        n_bins = int(np.round(t1_s * AP_Bin_Res))
        edges  = np.linspace(0.0, n_bins * Bin_size, n_bins + 1)
        AP_Bin = np.zeros((len(sp), n_bins), dtype=np.float32)
        for k, st in enumerate(sp):
            counts, _ = np.histogram(st, bins=edges)
            AP_Bin[k] = counts
        stim_sel = tr['whisker_stim_time'].notna() & tr['whisker_stim_amplitude'].isin(amps_stim)
        Stim     = {a: np.sort(tr.loc[stim_sel & (tr['whisker_stim_amplitude'] == a),
                                       'whisker_stim_time'].to_numpy()) for a in amps_stim}
        if 'no_stim' in tr.columns and 'no_stim_time' in tr.columns:
            catch_sel = tr['no_stim'].astype(bool) & tr['no_stim_time'].notna()
            Catch     = np.sort(tr.loc[catch_sel, 'no_stim_time'].to_numpy())
        else:
            Catch = np.array([])
        CATCH_m, _ = event_triggered_mean(AP_Bin, Catch, AP_Bin_Res, pre_bins, post_bins)
        STM4_m, _  = event_triggered_mean(AP_Bin, Stim[4], AP_Bin_Res, pre_bins, post_bins)
        STM_vis    = {a: None for a in amps_stim}
        for a in amps_stim:
            STM_vis[a], _ = event_triggered_mean(AP_Bin, Stim[a], AP_Bin_Res, pre_bins, post_bins)
        score, t_sig, _ = compute_cd_score(
            AP_Bin, STM_vis, CATCH_m, Stim, Catch, AP_Bin_Res,
            method=CD_METHOD, amp=CD_USE_AMP, win=CD_WINDOW, wins=CD_WINDOWS_2,
            pre_baseline=CD_PRE_BASELINE, matrix_step_bins=CD_MATRIX_STEP_BINS,
            matrix_zscore=CD_MATRIX_ZSCORE)
        in_range = (t_sig >= t0_s) & (t_sig <= t1_s)
        score_in = score[in_range]
        ths_pct  = np.percentile(score_in, pct_axis)
        if score_in.max() > ths_pct[-1]:
            ths = np.concatenate([ths_pct, np.linspace(ths_pct[-1], score_in.max(), 20)[1:]])
        else:
            ths = ths_pct
        all_stim = np.sort(np.concatenate([Stim[a] for a in amps_stim]))
        tp4, fp_ = [], []
        for thr in ths:
            det  = detect_peaks_fn(score, t_sig, thr, refractory_s, Bin_size, t0_s, t1_s)
            in_w = np.zeros(len(det), bool)
            for st in all_stim: in_w |= (det >= st) & (det <= st + tol_post)
            fp_.append(np.sum(~in_w) / max(len(det), 1))
            tp4.append(sum(np.any((det >= st) & (det <= st + tol_post)) for st in Stim[4])
                       / max(len(Stim[4]), 1))
        best_i   = np.argmax(np.array(tp4) - np.array(fp_))
        best_thr = ths[best_i]
        det_best = detect_peaks_fn(score, t_sig, best_thr, refractory_s, Bin_size, t0_s, t1_s)
        lick_times = np.array([])
        if has_piezo:
            pr  = 1.0 / np.median(np.diff(piezo_ts))
            env = gaussian_filter1d(np.abs(piezo_data), sigma=int(0.01 * pr))
            med = np.median(env); mad = np.median(np.abs(env - med))
            pidx, _ = find_peaks(env, height=med + 15.0 * mad, distance=int(0.15 * pr))
            lick_times = piezo_ts[pidx]
        print(f'  → {len(det_best)} detections  |  {len(lick_times)} licks')
        return dict(AP_Bin=AP_Bin, Stim=Stim, Catch=Catch,
                    score=score, t_sig=t_sig, best_thr=best_thr, det_best=det_best,
                    has_piezo=has_piezo, piezo_data=piezo_data, piezo_ts=piezo_ts,
                    lick_times=lick_times, t0_s=t0_s, t1_s=t1_s)

    def plot_session_vis(session=session_names_vis[0], t0=200.0, t1=300.0):
        if session not in _vis_cache:
            _vis_cache[session] = _compute_vis(session)
        d = _vis_cache[session]
        mask_s  = (d['t_sig'] >= t0) & (d['t_sig'] <= t1)
        det_win = d['det_best'][(d['det_best'] >= t0) & (d['det_best'] <= t1)]
        b0v     = int(t0 * AP_Bin_Res)
        b1v     = min(int(t1 * AP_Bin_Res), d['AP_Bin'].shape[1])
        t_rate  = np.arange(b0v, b1v) * Bin_size
        rate_sm = gaussian_filter1d(d['AP_Bin'][:, b0v:b1v].mean(axis=0), sigma=3)
        n_panels = 3 if d['has_piezo'] else 2
        ratios   = [2, 1.5, 1.5] if d['has_piezo'] else [2, 1.5]
        fig, axes = plt.subplots(n_panels, 1, figsize=(18, 9 if d['has_piezo'] else 6),
                                 sharex=True, gridspec_kw={'height_ratios': ratios})
        leg_common = [
            Line2D([0],[0], color='navy',      lw=1.5,                 label='CD Score'),
            Line2D([0],[0], color='gray',      lw=1.5, linestyle='--', label=f'Threshold ({d["best_thr"]:.3f})'),
            Line2D([0],[0], color='cyan',      lw=1.5, linestyle=':',  label='Detection'),
            Line2D([0],[0], color='steelblue', lw=1.5, linestyle='--', label='Catch'),
        ] + [Line2D([0],[0], color=colors_amp[a], lw=1.5, label=f'Stim amp{a}') for a in amps_stim]
        ax = axes[0]
        ax.plot(d['t_sig'][mask_s], d['score'][mask_s], color='navy', lw=0.7)
        ax.axhline(d['best_thr'], color='gray', lw=1.0, linestyle='--')
        for dt in det_win:
            ax.axvline(dt, color='cyan', alpha=0.8, lw=1.2, linestyle=':')
        for a in amps_stim:
            for st in d['Stim'][a]:
                if t0 <= st <= t1: ax.axvline(st, color=colors_amp[a], alpha=0.7, lw=1.2)
        for ct in d['Catch']:
            if t0 <= ct <= t1: ax.axvline(ct, color='steelblue', alpha=0.4, lw=1.0, linestyle='--')
        ax.legend(handles=leg_common, fontsize=8, loc='upper right')
        ax.set_ylabel('CD Score')
        ax.set_title(f'{session} — {t0:.0f}–{t1:.0f} s  |  {len(det_win)} detections  [{CD_METHOD}]')
        ax2 = axes[1]
        ax2.fill_between(t_rate, rate_sm, alpha=0.7, color='gray')
        for dt in det_win:
            ax2.axvline(dt, color='cyan', alpha=0.8, lw=1.2, linestyle=':')
        for a in amps_stim:
            for st in d['Stim'][a]:
                if t0 <= st <= t1: ax2.axvline(st, color=colors_amp[a], alpha=0.7, lw=1.2)
        for ct in d['Catch']:
            if t0 <= ct <= t1: ax2.axvline(ct, color='steelblue', alpha=0.4, lw=1.0, linestyle='--')
        fr_leg = [Line2D([0],[0], color='gray', lw=8, alpha=0.7, label=f'Firing rate ({THIS_AREA})'),
                  Line2D([0],[0], color='cyan', lw=1.5, linestyle=':', label='Detection'),
                  Line2D([0],[0], color='steelblue', lw=1.5, linestyle='--', label='Catch'),
                  ] + [Line2D([0],[0], color=colors_amp[a], lw=1.5, label=f'Stim amp{a}') for a in amps_stim]
        ax2.legend(handles=fr_leg, fontsize=8, loc='upper right')
        ax2.set_ylabel('Spikes/bin\n(pop. mean)')
        if d['has_piezo']:
            ax3 = axes[2]
            mp   = (d['piezo_ts'] >= t0) & (d['piezo_ts'] <= t1)
            lw_  = d['lick_times'][(d['lick_times'] >= t0) & (d['lick_times'] <= t1)]
            ax3.plot(d['piezo_ts'][mp], d['piezo_data'][mp], color='darkorange', lw=0.5, label='PiezoLickSignal')
            for dt in det_win:
                ax3.axvline(dt, color='cyan', alpha=0.8, lw=1.2, linestyle=':')
            if mp.any():
                ax3.scatter(lw_, np.zeros(len(lw_)), color='red', s=12, zorder=6,
                            marker='o', label=f'Lick (n={len(lw_)})')
            for a in amps_stim:
                for st in d['Stim'][a]:
                    if t0 <= st <= t1: ax3.axvline(st, color=colors_amp[a], alpha=0.6, lw=1.2)
            for ct in d['Catch']:
                if t0 <= ct <= t1: ax3.axvline(ct, color='steelblue', alpha=0.4, lw=1.0, linestyle='--')
            ax3.set_ylabel('Piezo (V)')
            ax3.legend(fontsize=8, loc='upper right')
        axes[-1].set_xlabel('Time (s)')
        plt.tight_layout()
        plt.show()

    widgets.interact(
        plot_session_vis,
        session=widgets.Dropdown(options=session_names_vis, description='Session:'),
        t0=widgets.FloatText(value=200.0, description='t start (s):'),
        t1=widgets.FloatText(value=300.0, description='t end (s):'),
    )

interactive(children=(Dropdown(description='Session:', options=('AO026_20181122_180943.nwb', 'AO027_20181101_1…

In [56]:
# ── Plot 3 : P(lick) per condition (det/missed/burst) — heatmap + barplot ─
if not results:
    print('⚠️  Aucune session chargée.')
else:
    session_names_p3 = ['Mean'] + [r['path'].split('/')[-1] for r in results]
    row_labels_plot = [f'Amp{a} detected' for a in [4,3,2,1]] + \
                      [f'Amp{a} missed'   for a in [4,3,2,1]] + ['Spontaneous burst']
    colors_rows = ['#cc0000','#cc4400','#cc6600','#cc9900',
                   '#ff9999','#ffbb88','#ffcc88','#ffee99','steelblue']

    def get_prop_row(r):
        row = []
        for a in [4, 3, 2, 1]:
            tl, tnl, ml, mnl = r['stim_results'][a]
            row.append(tl / max(tl + tnl, 1))
        for a in [4, 3, 2, 1]:
            tl, tnl, ml, mnl = r['stim_results'][a]
            row.append(ml / max(ml + mnl, 1))
        row.append(r['burst_lick'] / max(r['burst_lick'] + r['burst_nolick'], 1))
        return row

    prop_per_session = [get_prop_row(r) for r in results]
    prop_arr = np.array(prop_per_session)

    def plot_plick(session='Mean'):
        fig, axes = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={'width_ratios': [1, 1.8]})
        if session == 'Mean':
            m_prop    = prop_arr.mean(axis=0)
            ci_prop   = CI95(prop_arr)
            title_suf = f'{N} sessions (mean ± 95% CI)'
            show_pts  = True
        else:
            idx       = session_names_p3.index(session) - 1
            m_prop    = np.array(get_prop_row(results[idx]))
            ci_prop   = np.zeros(len(m_prop))
            title_suf = session
            show_pts  = False
        im = axes[0].imshow(m_prop[:, np.newaxis], cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
        axes[0].set_xticks([0]); axes[0].set_xticklabels(['P(lick)'])
        axes[0].set_yticks(range(len(row_labels_plot)))
        axes[0].set_yticklabels(row_labels_plot, fontsize=9)
        for i, (m, ci) in enumerate(zip(m_prop, ci_prop)):
            txt = f'{m:.2f}\n±{ci:.2f}' if show_pts else f'{m:.2f}'
            axes[0].text(0, i, txt, ha='center', va='center', fontsize=8)
        axes[0].set_title(f'P(lick)\n({title_suf})')
        plt.colorbar(im, ax=axes[0], shrink=0.6)
        y_pos = np.arange(len(row_labels_plot))
        if show_pts:
            axes[1].barh(y_pos, m_prop, xerr=ci_prop, color=colors_rows,
                         alpha=0.8, edgecolor='k', linewidth=0.5, capsize=4)
            rng = np.random.default_rng(42)
            for i in range(len(row_labels_plot)):
                jitter = rng.uniform(-0.25, 0.25, N)
                axes[1].scatter(prop_arr[:, i], np.full(N, i) + jitter,
                                color='k', s=20, zorder=5, alpha=0.7)
            axes[1].set_title('P(lick) per condition (mean ± 95% CI,  • = individual session)')
        else:
            axes[1].barh(y_pos, m_prop, color=colors_rows, alpha=0.8, edgecolor='k', linewidth=0.5)
            axes[1].set_title(f'P(lick) — {session}')
        axes[1].set_yticks(y_pos)
        axes[1].set_yticklabels(row_labels_plot, fontsize=9)
        axes[1].set_xlim(0, 1.1)
        axes[1].set_xlabel('P(lick within 1s)')
        axes[1].axvline(0.5, color='gray', linestyle='--', lw=1)
        plt.suptitle(f'Burst/stim discrimination — {title_suf}', fontweight='bold')
        plt.tight_layout()
        plt.show()

    widgets.interact(plot_plick, session=session_names_p3)

interactive(children=(Dropdown(description='session', options=('Mean', 'AO026_20181122_180943.nwb', 'AO027_201…

In [57]:
# ── Plot 4 : Chance vs Burst vs Stim4 + Δ(Burst−Chance) ──────────────────
if not results:
    print('⚠️  Aucune session chargée.')
else:
    session_names_p4 = ['Mean'] + [r['path'].split('/')[-1] for r in results]
    chance_arr = np.array([r['chance_lick'] for r in results])
    burst_arr  = np.array([r['p_burst']     for r in results])
    stim4_arr  = np.array([r['p_stim4']     for r in results])
    diff_arr   = burst_arr - chance_arr
    cats_bar   = ['Chance', 'Spontaneous burst', 'Stim4 detected', 'Δ Burst−Chance']
    colors_bar = ['lightgray', 'steelblue', 'red', 'mediumseagreen']

    def plot_chance_burst(session='Mean'):
        fig, ax = plt.subplots(figsize=(8, 5))
        if session == 'Mean':
            arrs  = [chance_arr, burst_arr, stim4_arr, diff_arr]
            means = [np.nanmean(a) for a in arrs]
            cis   = [1.96 * np.nanstd(a, ddof=1) / np.sqrt(np.sum(~np.isnan(a))) for a in arrs]
            bars  = ax.bar(cats_bar, means, yerr=cis, color=colors_bar,
                           edgecolor='k', linewidth=0.8, capsize=7, alpha=0.85)
            rng = np.random.default_rng(42)
            for i, arr in enumerate(arrs):
                valid = arr[~np.isnan(arr)]
                ax.scatter(np.full(len(valid), i) + rng.uniform(-0.15, 0.15, len(valid)),
                           valid, color='k', s=30, zorder=5, alpha=0.8)
            for bar, m in zip(bars, means):
                ypos = bar.get_height() + 0.03 if m >= 0 else bar.get_height() - 0.06
                ax.text(bar.get_x() + bar.get_width()/2, ypos,
                        f'{m:+.2f}' if 'Δ' in bar.get_label() else f'{m:.2f}',
                        ha='center', fontsize=10, fontweight='bold')
            ax.axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
            ax.set_title(f'Chance vs Spontaneous burst vs Stim4\n{N} sessions (mean ± 95% CI,  • = session)')
        else:
            idx  = session_names_p4.index(session) - 1
            r    = results[idx]
            vals = [r['chance_lick'], r['p_burst'], r['p_stim4'], r['p_burst'] - r['chance_lick']]
            bars = ax.bar(cats_bar, vals, color=colors_bar, edgecolor='k', linewidth=0.8, alpha=0.85)
            for bar, v in zip(bars, vals):
                ypos = v + 0.03 if v >= 0 else v - 0.06
                ax.text(bar.get_x() + bar.get_width()/2, ypos,
                        f'{v:+.3f}' if v == vals[-1] else f'{v:.3f}',
                        ha='center', fontsize=10, fontweight='bold')
            ax.axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
            ax.set_title(f'Chance vs Burst vs Stim4 — {session}\n(N bursts = N ctrl chance = {r["n_burst"]})')
        ax.set_ylim(-0.3, 1.2)
        ax.set_ylabel(f'P(lick within {lick_window_s:.0f}s)')
        plt.tight_layout()
        plt.show()

    widgets.interact(plot_chance_burst, session=session_names_p4)

interactive(children=(Dropdown(description='session', options=('Mean', 'AO026_20181122_180943.nwb', 'AO027_201…

In [58]:
# ── Statistical tests — interactive plot ──────────────────────────────────
from scipy.stats import fisher_exact

if not results:
    print('⚠️  Aucune session chargée.')
else:
    session_names_stat = ['Mean'] + [r['path'].split('/')[-1] for r in results]

    def _wilcoxon(a, b):
        valid = ~np.isnan(a) & ~np.isnan(b)
        n = valid.sum()
        if n < 3:
            return None, None, f'n.s. (n={n} < 3)'
        stat, p = wilcoxon(a[valid], b[valid], alternative='greater')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
        return stat, p, sig

    def _sig(p):
        return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

    def _fisher_pooled():
        b_l  = sum(r['burst_lick']   for r in results)
        b_nl = sum(r['burst_nolick'] for r in results)
        c_l  = sum(round(r['chance_lick'] * r['n_burst']) for r in results if not np.isnan(r['chance_lick']))
        c_nl = sum(r['n_burst'] for r in results) - c_l
        s_l  = sum(r['stim_results'][4][0] for r in results)
        s_nl = sum(r['stim_results'][4][1] for r in results)
        _, p_bc  = fisher_exact([[b_l, b_nl], [c_l,  c_nl]], alternative='greater')
        _, p_s4b = fisher_exact([[s_l, s_nl], [b_l,  b_nl]], alternative='greater')
        _, p_s4c = fisher_exact([[s_l, s_nl], [c_l,  c_nl]], alternative='greater')
        return [p_bc, p_s4b, p_s4c]

    def _fisher_session(r):
        b_l  = r['burst_lick']; b_nl = r['burst_nolick']
        c_l  = round(r['chance_lick'] * r['n_burst']); c_nl = r['n_burst'] - c_l
        s_l  = r['stim_results'][4][0]; s_nl = r['stim_results'][4][1]
        _, p_bc  = fisher_exact([[b_l, b_nl], [c_l,  c_nl]], alternative='greater')
        _, p_s4b = fisher_exact([[s_l, s_nl], [b_l,  b_nl]], alternative='greater')
        _, p_s4c = fisher_exact([[s_l, s_nl], [c_l,  c_nl]], alternative='greater')
        return [p_bc, p_s4b, p_s4c], dict(b_l=b_l, b_nl=b_nl, c_l=c_l, c_nl=c_nl, s_l=s_l, s_nl=s_nl)

    FISHER_PVALS = _fisher_pooled()

    def _draw_sig_bars(ax, log_p, p_vals, sigs, means_diff, y_max, title, annot_color):
        bar_colors = ['mediumseagreen' if (p is not None and p < 0.05) else 'lightcoral' for p in p_vals]
        thr = -np.log10(0.05)
        ax.bar(range(3), log_p, color=bar_colors, edgecolor='k', lw=0.8, alpha=0.85)
        ax.axhline(thr, color='gray', lw=1.5, linestyle='--', label='p = 0.05')
        ax.set_xticks(range(3))
        ax.set_xticklabels(['Burst\n> Chance', 'Stim4\n> Burst', 'Stim4\n> Chance'], fontsize=9)
        ax.set_ylabel('−log₁₀(p-value)')
        ax.set_title(title)
        y_scale = max(y_max, thr)
        ax.set_ylim(0, y_scale * 1.65)
        ax.legend(fontsize=8)
        for i, (lp, p, sig) in enumerate(zip(log_p, p_vals, sigs)):
            md_str = f'\nΔ={means_diff[i]:+.3f}' if means_diff is not None else ''
            ann = (f'p={p:.4f}\n{sig}{md_str}' if (p is not None and p >= 0.001)
                   else f'p={p:.2e}\n{sig}{md_str}' if p is not None else sig)
            y_ann = lp + y_scale * 0.04 if lp > 0 else y_scale * 0.04
            ax.text(i, y_ann, ann, ha='center', va='bottom', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=annot_color, alpha=0.85, edgecolor='none'))

    def plot_stats(session='Mean'):
        test_pairs = [(burst_arr, chance_arr), (stim4_arr, burst_arr), (stim4_arr, chance_arr)]
        if session == 'Mean':
            fig, (ax, ax_w, ax_f) = plt.subplots(1, 3, figsize=(18, 5),
                                                   gridspec_kw={'width_ratios': [1.2, 1, 1]})
            fig.suptitle('Statistical tests — Chance / Burst / Stim4', fontweight='bold')
            sess_labels  = [r['path'].split('/')[-1][:18] for r in results]
            colors_sess  = plt.cm.tab10(np.linspace(0, 1, N))
            for c, b, col, lbl in zip(chance_arr, burst_arr, colors_sess, sess_labels):
                ax.plot([0, 1], [c, b], color=col, lw=1.2, alpha=0.6)
                ax.scatter([0, 1], [c, b], color=col, s=60, zorder=5, label=lbl)
            for xi, arr in [(0, chance_arr), (1, burst_arr)]:
                m  = np.nanmean(arr)
                ci = 1.96 * np.nanstd(arr, ddof=1) / np.sqrt(np.sum(~np.isnan(arr)))
                ax.errorbar(xi, m, yerr=ci, fmt='D', color='black', markersize=10, capsize=6, lw=2.5, zorder=10)
            ax.set_xticks([0, 1]); ax.set_xticklabels(['Chance', 'Spontaneous burst'], fontsize=11)
            ax.set_ylabel('P(lick within 1s)'); ax.set_ylim(-0.05, 1.25)
            ax.set_title('Paired sessions: Chance vs Burst')
            ax.legend(fontsize=7, loc='upper left', ncol=2)
            p_w, sigs_w, diffs_w = [], [], []
            for a, b in test_pairs:
                _, p, sig = _wilcoxon(a, b)
                p_w.append(p); sigs_w.append(sig); diffs_w.append(np.nanmean(a - b))
            log_w = [-np.log10(p) if p is not None and p > 0 else 0 for p in p_w]
            _draw_sig_bars(ax_w, log_w, p_w, sigs_w, diffs_w, max(log_w) if log_w else 1,
                           f'Wilcoxon — session-level  (n={N})', 'white')
            sigs_f  = [_sig(p) for p in FISHER_PVALS]
            diffs_f = [np.nanmean(a - b) for a, b in test_pairs]
            log_f   = [-np.log10(p) if p > 0 else 0 for p in FISHER_PVALS]
            _draw_sig_bars(ax_f, log_f, FISHER_PVALS, sigs_f, diffs_f, max(log_f) if log_f else 1,
                           'Fisher exact — pooled events  (all sessions)', '#ffffcc')
        else:
            idx = session_names_stat.index(session) - 1
            r   = results[idx]
            c_val, b_val, s_val = r['chance_lick'], r['p_burst'], r['p_stim4']
            d_val = b_val - c_val; n_b = r['n_burst']
            fp, fc = _fisher_session(r)
            sigs_f = [_sig(p) for p in fp]
            log_f  = [-np.log10(p) if p > 0 else 0 for p in fp]
            fig, (ax, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5),
                                                gridspec_kw={'width_ratios': [1, 1, 1]})
            fig.suptitle(f'Statistical tests — {session}', fontweight='bold')
            bars = ax.bar(['Chance', 'Burst', 'Stim4'], [c_val, b_val, s_val],
                          color=['lightgray', 'steelblue', 'red'], edgecolor='k', lw=0.8, alpha=0.85)
            for bar, v in zip(bars, [c_val, b_val, s_val]):
                ax.text(bar.get_x() + bar.get_width()/2, v + 0.02,
                        f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
            ax.set_ylim(0, 1.2); ax.set_ylabel('P(lick within 1s)')
            ax.set_title(f'P(lick) — {session[:30]}')
            col_d = 'mediumseagreen' if d_val > 0 else 'lightcoral'
            ax2.bar(['Δ Burst − Chance'], [d_val], color=col_d, edgecolor='k', lw=0.8, alpha=0.85)
            ax2.axhline(0, color='k', lw=1)
            ax2.text(0, d_val + (0.02 if d_val >= 0 else -0.07), f'{d_val:+.3f}',
                     ha='center', fontsize=14, fontweight='bold')
            ax2.set_ylim(-0.5, 0.8); ax2.set_ylabel('Δ P(lick)')
            info = (f'N spontaneous bursts : {n_b}\n'
                    f'N chance controls    : {n_b}  (same N ✓)\n\n'
                    f'P(lick | burst)  = {b_val:.3f}\n'
                    f'P(lick | chance) = {c_val:.3f}\n'
                    f'Δ                = {d_val:+.3f}')
            ax2.text(1.05, 0.5, info, transform=ax2.transAxes, fontsize=10, va='center', family='monospace',
                     bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
            ax2.set_title('Burst − Chance')
            _draw_sig_bars(ax3, log_f, fp, sigs_f,
                           [b_val - c_val, s_val - b_val, s_val - c_val], max(log_f) if log_f else 1,
                           f'Fisher exact — this session\n(burst: {fc["b_l"]}L / {fc["b_nl"]}NL   chance: {fc["c_l"]}L / {fc["c_nl"]}NL)',
                           '#ffffcc')
        plt.tight_layout()
        plt.show()

    widgets.interact(plot_stats, session=session_names_stat)

interactive(children=(Dropdown(description='session', options=('Mean', 'AO026_20181122_180943.nwb', 'AO027_201…